# Human evaluation for all three parts (Team 16)

CPU notebook. It reads the rating files that each person downloaded from the rating pages (`scripts/make_rating_pages.py`) and computes every human-evaluation number in the report:

- Part 1: groundedness 1 to 5 for the 25 final golden-set answers, by both members (`eval/human/p1_rater_*.csv`).
- Part 2: blinded comparison of 25 image pairs, final run (LoRA r16) against the base model, by at least three raters (`eval/human/p2_rater_*.csv`). The left/right order is rebuilt here from seed 266, so the rating page never showed which side was which.
- Part 3: pass/fail of the 15 scenarios by both members (`eval/p3_judging_*.csv`), agreement, the resolved decision (`eval/p3_judging_resolved.csv`), task success with and without the critique loop, and critic agreement.

In [1]:
import os, subprocess
REPO = "github.com/sadaffatimaee/data266-lab2-team16.git"
if subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True).returncode != 0 and os.path.isdir("/kaggle"):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        subprocess.run(["git", "clone", "-q", f"https://{token}@{REPO}", "/tmp/repo"], check=True)
        subprocess.run(["git", "-C", "/tmp/repo", "remote", "set-url", "origin", f"https://{REPO}"], check=True)
        os.chdir("/tmp/repo")
    except Exception as e:
        print("repository not cloned:", type(e).__name__)
ROOT = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()

import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
"gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

{'commit': 'bff4557', 'python': '3.12.10', 'torch': '2.14.0+cpu', 'gpu': 'cpu', 'seed': 266}


In [2]:
import os, glob, json, random
import pandas as pd
EV = os.path.join(ROOT, "eval")
HUM = os.path.join(EV, "human")
def raters(prefix):
    return {os.path.basename(f)[len(prefix):-4]: pd.read_csv(f, dtype=str, keep_default_na=False)
            for f in sorted(glob.glob(os.path.join(HUM, f"{prefix}*.csv")))}
out = {}

## Part 1: groundedness

In [3]:
p1 = raters("p1_rater_")
print("Part 1 raters:", list(p1))
if len(p1) >= 2:
    names = list(p1)[:2]
    m = p1[names[0]][["qid", "groundedness"]].merge(p1[names[1]][["qid", "groundedness"]], on="qid", suffixes=("_a", "_b"))
    a, b = m["groundedness_a"].astype(float), m["groundedness_b"].astype(float)
    res = {"questions": len(m), "mean groundedness (both raters)": round(float(pd.concat([a, b]).mean()), 2),
           f"mean {names[0]}": round(float(a.mean()), 2), f"mean {names[1]}": round(float(b.mean()), 2),
           "agreement within 1 point": round(float(((a - b).abs() <= 1).mean()), 3),
           "exact agreement": round(float((a == b).mean()), 3)}
    out["p1"] = res
    print(res)
    print("answers with a gap of 2 or more points:", list(m[(a - b).abs() >= 2]["qid"]))
else:
    print("waiting for two rating files")

Part 1 raters: []
waiting for two rating files


## Part 2: blinded pairwise comparison

In [4]:
from huggingface_hub import hf_hub_download
meta = pd.read_csv(hf_hub_download("sadaffatimaeee/naruto-lora-generations", "metadata.csv", repo_type="dataset"))
keys = sorted({(r.pid, int(r.seed), r.prompt) for r in meta.itertuples()})
rng = random.Random(SEED)
chosen = rng.sample(keys, 25)
key = []
for i, (pid, seed, prompt) in enumerate(chosen, 1):
    final_left = rng.random() < 0.5
    key.append({"pair": f"Q{i:02d}", "pid": pid, "seed": seed, "type": meta[meta["pid"] == pid]["type"].iloc[0], "final_side": "L" if final_left else "R"})
key = pd.DataFrame(key)
print(key.groupby("type").size().to_dict(), "| final model on the left in", int((key["final_side"] == "L").sum()), "of 25 pairs")

p2 = raters("p2_rater_")
print("Part 2 raters:", list(p2))
if p2:
    long = []
    for name, d in p2.items():
        d = d.merge(key, on="pair")
        for crit in ("style", "prompt_match"):
            for _, r in d.iterrows():
                if r[crit] in ("L", "R"):
                    long.append({"rater": name, "pair": r["pair"], "type": r["type"], "criterion": crit, "final_wins": r[crit] == r["final_side"]})
    long = pd.DataFrame(long)
    per_rater = long.pivot_table(index="rater", columns="criterion", values="final_wins", aggfunc="mean").round(3)
    print("win rate of the final run, per rater")
    print(per_rater.to_string())
    pooled = long.groupby("criterion")["final_wins"].mean().round(3)
    by_type = long.pivot_table(index="type", columns="criterion", values="final_wins", aggfunc="mean").round(3)
    maj = long.groupby(["criterion", "pair"])["final_wins"].mean().reset_index()
    maj_rate = maj.assign(win=maj["final_wins"] > 0.5).groupby("criterion")["win"].mean().round(3)

    def fleiss(crit):
        t = long[long["criterion"] == crit].pivot_table(index="pair", columns="rater", values="final_wins", aggfunc="first").dropna()
        n = t.shape[1]
        if n < 2 or len(t) == 0:
            return None
        cnt = np.stack([t.sum(1).values, n - t.sum(1).values], 1).astype(float)
        P = ((cnt ** 2).sum(1) - n) / (n * (n - 1))
        pj = cnt.sum(0) / cnt.sum()
        pe = (pj ** 2).sum()
        return round(float((P.mean() - pe) / (1 - pe)), 3) if pe < 1 else None

    res = {"raters": len(p2), "pooled win rate": pooled.to_dict(), "majority-vote win rate": maj_rate.to_dict(),
           "win rate by prompt type": by_type.to_dict(), "Fleiss kappa": {c: fleiss(c) for c in ("style", "prompt_match")}}
    out["p2"] = res
    print(json.dumps(res, indent=1))
    if len(p2) < 3:
        print("note: the brief asks for at least 3 raters (both members and one outside person)")
else:
    print("waiting for rating files")

{'compositional': 7, 'in_domain': 13, 'out_of_domain': 5} | final model on the left in 15 of 25 pairs
Part 2 raters: []
waiting for rating files


## Part 3: scenario judging, agreement and critic agreement

In [5]:
def pf(d):
    d = d.copy()
    d.columns = ["sid", "a", "b", "first"] + list(d.columns[4:])
    return d.set_index("sid")

judges = {os.path.basename(f)[11:-4]: pf(pd.read_csv(f, dtype=str, keep_default_na=False))
          for f in sorted(glob.glob(os.path.join(EV, "p3_judging_*.csv"))) if not f.endswith("_resolved.csv")}
print("Part 3 judges:", list(judges))
for n, d in judges.items():
    print(f"  {n}: pass without critique {int((d['a'] == 'P').sum())}/15, with critique {int((d['b'] == 'P').sum())}/15")
res = {}
if len(judges) >= 2:
    (n1, d1), (n2, d2) = list(judges.items())[:2]
    agree = {c: float((d1[c] == d2[c]).mean()) for c in ("a", "b")}
    both = pd.concat([d1[c] == d2[c] for c in ("a", "b")]).mean()
    res["judge agreement"] = {"without critique": round(agree["a"], 3), "with critique": round(agree["b"], 3), "all 30 decisions": round(float(both), 3)}
    dis = [(s, c) for c in ("a", "b") for s in d1.index if d1.loc[s, c] != d2.loc[s, c]]
    print("disagreements (scenario, A=without/B=with):", [(s, c.upper()) for s, c in dis])

rpath = os.path.join(EV, "p3_judging_resolved.csv")
final = None
if os.path.exists(rpath) and len(judges) >= 2:
    final = pf(pd.read_csv(rpath, dtype=str, keep_default_na=False))
    if (final[["a", "b"]] == "").any().any():
        final = None
if final is not None:
    res["task success (resolved)"] = {"without critique": int((final["a"] == "P").sum()), "with critique": int((final["b"] == "P").sum())}
    auto = pd.read_csv(os.path.join(EV, "p3_results.csv"), keep_default_na=False)
    crit = auto[(auto["kind"] == "scenario") & (auto["critique"].astype(str) == "True")].set_index("id")
    rows = []
    for sid, r in crit.iterrows():
        dec = r["critic_decisions"].split("|")[0] if r["critic_decisions"] else ""
        human = final.loc[sid, "first"]
        if dec and human in ("Y", "N"):
            rows.append({"sid": sid, "critic": dec, "human": "accept" if human == "Y" else "revise"})
    ca = pd.DataFrame(rows)
    if len(ca):
        ca["match"] = ca["critic"] == ca["human"]
        print(ca.to_string(index=False))
        res["critic agreement"] = {"matches": int(ca["match"].sum()), "of": len(ca), "rate": round(float(ca["match"].mean()), 3)}
else:
    print("waiting for both judging sheets and the resolved sheet")
out["p3"] = res
print(json.dumps(res, indent=1))
print("RESULTS_JSON", json.dumps(out))

Part 3 judges: ['sadaf']
  sadaf: pass without critique 8/15, with critique 10/15
waiting for both judging sheets and the resolved sheet
{}
RESULTS_JSON {"p3": {}}
